In [1]:
import json
import os
from typing import List, Optional
os.environ.pop("TRITON_PTXAS_PATH", None)
import librosa
import numpy as np
from vllm import LLM, SamplingParams

INFO 05-11 23:37:16 [__init__.py:239] Automatically detected platform cuda.


In [ ]:
# ── Fixed parameters (edit these before running) ──────────────────────────────
SUBSET_PATH       = "dcase_subset.json"          # path to sampled subset JSON
MODEL             = "Qwen/Qwen2.5-Omni-7B"       # vLLM model identifier
TEMPERATURE       = 0.2                           # sampling temperature
MAX_TOKENS        = 256                           # max generated tokens
OUTPUT_PREDICTIONS = "predictions.json"           # set to a .json path to save results, or None
SYSTEM_PROMPT     = (
    "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, "
    "capable of perceiving auditory and visual inputs, as well as generating text and speech."
)

In [3]:
llm = LLM(
    model=MODEL,
    max_model_len=20000,
    max_num_seqs=1,
    limit_mm_per_prompt={"audio": 4},
    trust_remote_code=True,
)



Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}
`torch_dtype` is deprecated! Use `dtype` instead!


INFO 05-11 23:37:20 [config.py:2968] Downcasting torch.float32 to torch.float16.
INFO 05-11 23:37:32 [config.py:717] This model supports multiple tasks: {'score', 'generate', 'classify', 'embed', 'reward'}. Defaulting to 'generate'.
WARNING 05-11 23:37:32 [arg_utils.py:1658] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 05-11 23:37:32 [llm_engine.py:240] Initializing a V0 LLM engine (v0.8.5.post1) with config: model='Qwen/Qwen2.5-Omni-7B', speculative_config=None, tokenizer='Qwen/Qwen2.5-Omni-7B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=20000, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='auto', 

Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]


INFO 05-11 23:37:43 [loader.py:458] Loading weights took 7.13 seconds
INFO 05-11 23:37:43 [model_runner.py:1140] Model loading took 16.7373 GiB and 8.410998 seconds


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


WARNING 05-11 23:37:51 [model_runner.py:1305] Computed max_num_seqs (min(1, 20000 // 20113)) to be less than 1. Setting it to the minimum value of 1.
WARNING 05-11 23:37:52 [profiling.py:245] The sequence length used for profiling (max_num_batched_tokens / max_num_seqs = 20000) is too short to hold the multi-modal embeddings in the worst case (20113 tokens in total, out of which {'audio': 3000, 'image': 16384, 'video': 729} are reserved for multi-modal embeddings). This may cause certain multi-modal inputs to fail during inference, even when the input text is short. To avoid this, you should increase `max_model_len`, reduce `max_num_seqs`, and/or reduce `mm_counts`.
INFO 05-11 23:38:08 [worker.py:287] Memory profiling takes 24.22 seconds
INFO 05-11 23:38:08 [worker.py:287] the current vLLM instance can use total_gpu_memory (31.74GiB) x gpu_memory_utilization (0.90) = 28.57GiB
INFO 05-11 23:38:08 [worker.py:287] model weights take 16.74GiB; non_torch_memory takes 0.09GiB; PyTorch activa

Capturing CUDA graph shapes:   0%|          | 0/1 [00:00<?, ?it/s]

INFO 05-11 23:38:12 [model_runner.py:1592] Graph capturing finished in 1 secs, took 0.05 GiB
INFO 05-11 23:38:12 [llm_engine.py:437] init engine (profile, create kv cache, warmup model) took 28.34 seconds


In [4]:
def load_audio(path: str, sr: int = 16000):
    y, _ = librosa.load(path, sr=sr, mono=True)
    return y.astype(np.float32), sr
    

def load_subset(subset_path: str) -> List[dict]:
    with open(subset_path, "r", encoding="utf-8") as f:
        subset = json.load(f)
    if not isinstance(subset, list):
        raise ValueError("Subset file must contain a JSON list.")
    return subset

In [5]:
sam_audio = None
import prompts
# tool_result_path = '/home/u1501463/tool_use_LALM/apply_tool_results_accelerated/dev_audio_0001459'
# tool_output, tool_audios_path = prompts.tool_chain_result_read.read_tool_chain(tool_result_path, '<AUDIO>')
# print(tool_output)
# print(tool_audios_path)


def build_prompt(question: str, choices: List[str], tool_results_path) -> str:
    choice_text = "\n".join(choices) if choices else ""
    audio_token = "<|audio_bos|><|AUDIO|><|audio_eos|>"
    tool_output, tool_audios_path = prompts.tool_chain_result_read.read_tool_chain(
        tool_chain_folder_path=tool_results_path,
        audio_token=audio_token
    )

    system_prompt, user_prompt = prompts.QA_prompt(
        question=question,
        options=choice_text,
        audio_token=audio_token,
        tool_results=tool_output,
        tools_list=None,
        tools_description=None
    )
    
    return (
        f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{user_prompt}"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
    ), tool_audios_path


In [ ]:
def evaluate(
    llm,
    subset_path: str,
    temperature: float,
    max_tokens: int,
    system_prompt: str,
    output_predictions,
) -> None:
    subset = load_subset(subset_path)
    os.environ.pop("TRITON_PTXAS_PATH", None)
    sampling_params = SamplingParams(temperature=temperature, max_tokens=max_tokens)

    results = []
    correct = 0

    for idx, item in enumerate(subset, start=1):
        question = item.get("question", "")
        choices = item.get("choice", [])
        answer = item.get("answer", "N/A")
        problem_id = item.get("id", f"sample_{idx}")
        audio_path = item.get("audio_path") or item.get("audio_url")

        if not audio_path:
            raise ValueError(f"No audio path found for subset item index {idx}")

        audio_data, sr = load_audio(audio_path)

        tool_output_path = os.path.join(
            '/home/u1501463/tool_use_LALM/apply_tool_results_accelerated',
            problem_id
        )

        prompt, audios_path = build_prompt(
            question=question,
            choices=choices,
            tool_results_path=tool_output_path
        )

        outputs = llm.generate(
            [
                {
                    "prompt": prompt,
                    "multi_modal_data": {"audio": [
                        (audio_data, sr)
                        ] + [(load_audio(p)) for p in audios_path]},
                }
            ],
            sampling_params=sampling_params,
        )

        pred = outputs[0].outputs[0].text.strip()
        is_correct = answer != "N/A" and answer.strip() in pred

        results.append(
            {
                "index": idx,
                "id": problem_id,
                "question": question,
                "choices": choices,
                "answer": answer,
                "prediction": pred,
                "correct": is_correct,
            }
        )
        if is_correct:
            correct += 1

        if output_predictions:
            with open(output_predictions, "w", encoding="utf-8") as f:
                json.dump(results, f, indent=2, ensure_ascii=False)

    accuracy = correct / len(results) if results else 0.0
    print("=== Evaluation Summary ===")
    print(f"Total samples: {len(results)}")
    print(f"Correct predictions: {correct}")
    print(f"Accuracy: {accuracy:.2%}")

    if output_predictions:
        print(f"Saved detailed predictions to: {output_predictions}")

In [11]:
evaluate(
    llm,
    subset_path=os.path.abspath(SUBSET_PATH),
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    system_prompt=SYSTEM_PROMPT,
    output_predictions=OUTPUT_PREDICTIONS,
)

Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

=== Evaluation Summary ===
Total samples: 90
Correct predictions: 34
Accuracy: 37.78%
